# Association-error analysis

This notebook reproduces a nested task-held-out comparison of false-split and false-merge rates. The only available positive label is a shared CTF task; it is a same-challenge proxy, not an attack-activity, campaign, or actor label. Human terminal capture and Agent tool-call capture are confounded with the source label in this release.

The analysis asks whether an execution-mode score changes proxy association errors beyond a behavior-only ranker. It does not claim to measure real APT association errors.

## Fixed evaluation protocol

- 177 non-empty traces from 40 benchmark-qualified CTF task groups (CC BY 4.0 pinned release).
- Three outer task-grouped folds; thresholds are selected only from three-fold task-held-out predictions inside each outer training set.
- Test pairs include all traces within each benchmark and held-out fold, with no execution-mode filter.
- Positive proxy: same benchmark and task. Negative proxy: different task in the same benchmark.
- Pair-type strata: H–H, H–A, and A–A. Errors are averaged per query, per task, and then equally across benchmarks.
- 5,000 benchmark-stratified task-cluster bootstrap draws. The intervals are conditional on the fitted outer folds.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'run_association_error_analysis.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'run_association_error_analysis.py').exists(), f'Cannot find repo root from {Path.cwd()}'
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])

## Recompute

The script writes machine-readable pair scores and task metrics, the bootstrap summary, threshold audit, and PDF/PNG figures. Pair outputs contain trace IDs and proxy labels but no raw commands.

In [ ]:
completed = subprocess.run(
    [str(ROOT / '.venv' / 'bin' / 'python'), 'scripts/run_association_error_analysis.py', '--bootstraps', '5000'],
    cwd=ROOT, check=True, text=True, capture_output=True
)
print(completed.stdout[-6000:])

In [ ]:
import pandas as pd
from IPython.display import display, Image

result = json.loads((ROOT / 'results' / 'association_error_analysis.json').read_text())
rows = []
for method, metrics in result['summary'].items():
    rows.append({
        'method': method,
        'false split': metrics['false_split_rate']['estimate'],
        'false split 95% CI': metrics['false_split_rate']['ci_95'],
        'cross-mode false split': metrics['false_split_human_agent']['estimate'],
        'cross-mode 95% CI': metrics['false_split_human_agent']['ci_95'],
        'false merge': metrics['false_merge_rate']['estimate'],
        'false merge 95% CI': metrics['false_merge_rate']['ci_95'],
    })
display(pd.DataFrame(rows).set_index('method').round(3))

In [ ]:
display(pd.DataFrame(result['pair_counts']))
delta_rows = []
for baseline, metrics in result['paired_deltas'].items():
    for metric in ['balanced_accuracy', 'false_split_human_agent', 'false_merge_human_agent']:
        item = metrics[metric]
        delta_rows.append({'comparison': f'mode_aware − {baseline}', 'metric': metric,
                          'delta': item[f'mode_aware_minus_{baseline}'], '95% CI': item['ci_95']})
display(pd.DataFrame(delta_rows).round(3))
display(Image(filename=str(ROOT / 'results' / 'figures' / 'association_error_rates.png')))

## Inspect held-out error cases

These rows are trace-pair examples under the CTF task proxy. They are useful for checking what the prototype links or misses, but not for claiming a real attack was split or merged.

In [ ]:
pairs = pd.read_csv(ROOT / 'results' / 'association_error_pair_scores.csv')
cross_mode_misses = pairs[
    (pairs['same_task_proxy_for_evaluation_only'] == 1)
    & (pairs['pair_type_for_evaluation_only'] == 'H-A')
    & (pairs['predicted_same_task_proxy_mode_aware'] == 0)
]
different_task_links = pairs[
    (pairs['same_task_proxy_for_evaluation_only'] == 0)
    & (pairs['predicted_same_task_proxy_mode_aware'] == 1)
]
display(cross_mode_misses[['benchmark', 'query_trace_id', 'candidate_trace_id', 'mode_aware', 'threshold_mode_aware']].head(12))
display(different_task_links[['benchmark', 'query_trace_id', 'candidate_trace_id', 'mode_aware', 'threshold_mode_aware']].head(12))